# Ensembl — Genome Annotation and Comparative Genomics

**Ensembl** is a genome browser and annotation database maintained jointly by EMBL-EBI and the Wellcome Sanger Institute. It provides comprehensive, automatically generated gene models for vertebrate genomes and is one of the world's primary sources of genomic annotation.

Annotation layers provided by Ensembl:

| Layer | Description |
|---|---|
| **Genes & transcripts** | Gene models with exon/intron structure, biotype classification (protein-coding, lncRNA, pseudogene, …) |
| **Translations** | Canonical and alternative protein sequences per transcript |
| **Regulatory features** | Promoters, enhancers, CTCF binding sites derived from ENCODE/Roadmap data |
| **Variation** | SNPs, indels, somatic mutations cross-referenced to dbSNP / ClinVar |
| **Comparative genomics** | Whole-genome alignments, synteny blocks, gene trees, and %identity across up to 350 species |
| **Functional annotation** | GO terms, OMIM disease links, pathway cross-references via Ensembl Xrefs |

**API base:** `https://rest.ensembl.org`  
**Rate limit:** 15 requests / second (REST); add ≥ 0.1 s delays between calls.

**Reference:** Martin et al. (2023), *Nucleic Acids Research*, Ensembl 2023

# TODO

* [x] **Ingest data**
    * [x] Implement `ensembl_get()` helper with rate-limit delay and JSON headers
    * [x] Fetch species list from `/info/species`, cache to `data/ensembl_species.json`, parse into Polars DataFrame
    * [x] Look up a panel of human cancer genes by symbol (TP53, BRCA1, BRCA2, EGFR, KRAS, MYC, PTEN, RB1), cache to `data/ensembl_cancer_genes.json`
    * [x] Parse gene lookup results into a Polars DataFrame with genomic coordinates, biotype, and transcript count
    * [x] For TP53, fetch all transcripts and build a transcript-level DataFrame (transcript_id, biotype, length, exon count, canonical flag)
* [ ] **Explore and clean**
    * [ ] Summarise species coverage by assembly level and taxonomic division
    * [ ] Compare gene structures across the cancer gene panel (length, exon density, transcript diversity)
    * [ ] Examine biotype distribution across TP53 transcripts
* [ ] **Comparative genomics**
    * [ ] Fetch orthologues for TP53 across vertebrates via `/homology/id/{id}`
    * [ ] Plot sequence identity vs. evolutionary distance
    * [ ] Discuss gene tree topology and duplication events
* [ ] **Variation**
    * [ ] Retrieve ClinVar/dbSNP variants in the TP53 locus via `/overlap/id/{id}?feature=variation`
    * [ ] Classify variants by consequence (missense, synonymous, splice, etc.)
    * [ ] Discuss SIFT / PolyPhen pathogenicity scores and their statistical basis
* [ ] **Regulatory context**
    * [ ] Fetch regulatory features overlapping cancer gene loci
    * [ ] Link promoter / enhancer annotations to gene expression data
* [ ] **Visualization**
    * [ ] Gene-structure diagram for TP53 transcripts (exons, UTRs, CDS) using Bokeh
    * [ ] Dot plot of transcript count vs. genomic span for the cancer gene panel
    * [ ] Circos-style chromosome band plot showing cancer gene positions
* [ ] **Statistical analysis**
    * [ ] Discuss transcript diversity metrics (number of transcripts, alternative splicing index)
    * [ ] Multiple-testing considerations when scanning the genome for regulatory variants

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 API Helper and Constants

In [ ]:
ENSEMBL_BASE = "https://rest.ensembl.org"   # REST API root
DATA_DIR = Path("data")                      # local cache directory
DATA_DIR.mkdir(exist_ok=True)

# Ensembl allows up to 15 requests/second; a 0.1 s sleep keeps us safely below
_RATE_DELAY = 0.1

def ensembl_get(endpoint: str, params: dict = None, content_type: str = "application/json"):
    """
    Send a GET request to the Ensembl REST API.

    Automatically attaches the required ``Content-Type`` / ``Accept`` headers
    and inserts a small inter-request delay to respect the 15 req/s rate limit.

    Parameters
    ----------
    endpoint : str
        API path relative to ``ENSEMBL_BASE`` (e.g. ``"info/species"``).
    params : dict, optional
        URL query parameters forwarded verbatim.
    content_type : str
        MIME type for both ``Content-Type`` and ``Accept`` headers.
        Use ``"application/json"`` (default) for structured data or
        ``"text/plain"`` for raw sequence endpoints.

    Returns
    -------
    requests.Response
        Raw response; caller is responsible for parsing (``resp.json()`` or
        ``resp.text``).

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{ENSEMBL_BASE}/{endpoint}"
    headers = {"Content-Type": content_type, "Accept": content_type}
    resp = requests.get(url, params=params or {}, headers=headers, timeout=30)
    resp.raise_for_status()
    time.sleep(_RATE_DELAY)   # respect Ensembl's 15 req/s limit
    return resp

# Quick connectivity check — fetch the current Ensembl release number
info = ensembl_get("info/data").json()
print(f"Ensembl releases available: {info.get('releases')}")

### 1.2 Fetch Species List

In [ ]:
SPECIES_CACHE = DATA_DIR / "ensembl_species.json"

def fetch_species(cache_path: Path = SPECIES_CACHE) -> list[dict]:
    """
    Fetch the full list of species supported by the Ensembl REST API.

    Results are cached to ``cache_path`` so subsequent runs load instantly
    without hitting the network.

    Parameters
    ----------
    cache_path : Path
        Filesystem path for the JSON cache file.

    Returns
    -------
    list[dict]
        Raw species records from the ``/info/species`` endpoint.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    print("Fetching species list from Ensembl REST API …")
    resp = ensembl_get("info/species")
    species = resp.json().get("species", [])
    cache_path.write_text(json.dumps(species))
    print(f"Cached {len(species)} species to {cache_path}")
    return species


def parse_species(raw: list[dict]) -> pl.DataFrame:
    """
    Parse raw Ensembl species records into a tidy Polars DataFrame.

    Parameters
    ----------
    raw : list[dict]
        Species dicts as returned by ``/info/species``.

    Returns
    -------
    pl.DataFrame
        One row per species with columns: species_name, common_name,
        taxon_id, assembly, genebuild.
    """
    rows = [
        {
            "species_name": s.get("name"),          # e.g. "homo_sapiens"
            "common_name":  s.get("common_name"),   # e.g. "Human"
            "taxon_id":     s.get("taxon_id"),       # NCBI taxon integer
            "assembly":     s.get("assembly"),       # e.g. "GRCh38"
            "genebuild":    s.get("genebuild"),      # e.g. "2014-07"
        }
        for s in raw
    ]
    return pl.DataFrame(rows).with_columns(
        pl.col("taxon_id").cast(pl.Int32, strict=False)
    )


species_raw = fetch_species()
species_df = parse_species(species_raw)

print(f"\nShape : {species_df.shape}")
print(f"dtypes:\n{species_df.dtypes}")
species_df.head(5)

### 1.3 Look Up Human Cancer Genes by Symbol

In [ ]:
CANCER_GENES = ["TP53", "BRCA1", "BRCA2", "EGFR", "KRAS", "MYC", "PTEN", "RB1"]
CANCER_GENES_CACHE = DATA_DIR / "ensembl_cancer_genes.json"

def fetch_cancer_genes(
    symbols: list[str],
    species: str = "homo_sapiens",
    cache_path: Path = CANCER_GENES_CACHE,
) -> list[dict]:
    """
    Look up a list of gene symbols in Ensembl via ``/lookup/symbol/{species}/{symbol}``.

    Each call uses ``expand=1`` to include the full list of transcripts in the
    response.  Results are cached as a single JSON file so the network is only
    hit once.

    Parameters
    ----------
    symbols : list[str]
        HGNC gene symbols to query (e.g. ``["TP53", "BRCA1"]``).
    species : str
        Ensembl species string (default ``"homo_sapiens"``).
    cache_path : Path
        Where to write/read the cached JSON.

    Returns
    -------
    list[dict]
        One dict per gene as returned by the Ensembl ``/lookup/symbol`` endpoint.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    results = []
    for symbol in symbols:
        print(f"  Fetching {symbol} …")
        resp = ensembl_get(
            f"lookup/symbol/{species}/{symbol}",
            params={"expand": 1},       # include transcripts in the response
        )
        results.append(resp.json())

    cache_path.write_text(json.dumps(results))
    print(f"\nCached {len(results)} genes to {cache_path}")
    return results


cancer_genes_raw = fetch_cancer_genes(CANCER_GENES)
print(f"\nFetched {len(cancer_genes_raw)} gene records")
# Show top-level keys of the first record so we know what fields are available
print(f"Keys: {list(cancer_genes_raw[0].keys())}")

### 1.4 Parse Cancer Genes into a Polars DataFrame

In [ ]:
def parse_cancer_genes(raw: list[dict]) -> pl.DataFrame:
    """
    Parse raw Ensembl gene records into a tidy Polars DataFrame.

    Parameters
    ----------
    raw : list[dict]
        Gene dicts as returned by ``/lookup/symbol`` with ``expand=1``.

    Returns
    -------
    pl.DataFrame
        One row per gene with columns: gene_id, symbol, chromosome,
        start, end, strand, biotype, description, transcript_count.
    """
    rows = []
    for g in raw:
        transcripts = g.get("Transcript") or []   # list present when expand=1
        rows.append({
            "gene_id":          g.get("id"),           # ENSG stable ID
            "symbol":           g.get("display_name"), # HGNC symbol
            "chromosome":       g.get("seq_region_name"),
            "start":            g.get("start"),        # 1-based genomic start
            "end":              g.get("end"),           # 1-based genomic end
            "strand":           g.get("strand"),        # +1 or -1
            "biotype":          g.get("biotype"),       # e.g. "protein_coding"
            "description":      g.get("description"),  # free-text description
            "transcript_count": len(transcripts),      # number of annotated transcripts
        })

    return pl.DataFrame(rows).with_columns([
        pl.col("start").cast(pl.Int64),
        pl.col("end").cast(pl.Int64),
        pl.col("strand").cast(pl.Int8),
        pl.col("transcript_count").cast(pl.Int32),
        # Compute genomic span in kb for convenience
        ((pl.col("end") - pl.col("start") + 1) / 1_000).round(1).alias("span_kb"),
    ])


cancer_genes_df = parse_cancer_genes(cancer_genes_raw)

print(f"Shape : {cancer_genes_df.shape}")
print(f"\ndtypes:")
print(cancer_genes_df.dtypes)
print()
cancer_genes_df.head(8)

### 1.5 TP53 Transcript-Level DataFrame

In [ ]:
def parse_transcripts(gene_record: dict) -> pl.DataFrame:
    """
    Build a transcript-level DataFrame from a single Ensembl gene record.

    Each transcript entry from ``expand=1`` contains its own ``Exon`` list;
    we count those to derive ``exon_count``.  The ``is_canonical`` flag is set
    to ``True`` for the single transcript designated by Ensembl as the best
    representative model (the one whose CDS is used for protein annotation).

    Parameters
    ----------
    gene_record : dict
        Single gene record from the Ensembl ``/lookup/symbol`` response with
        ``expand=1`` (i.e. containing a ``"Transcript"`` key).

    Returns
    -------
    pl.DataFrame
        One row per transcript with columns: transcript_id, name, biotype,
        length, exon_count, is_canonical.
    """
    transcripts = gene_record.get("Transcript") or []
    rows = []
    for t in transcripts:
        exons = t.get("Exon") or []
        rows.append({
            "transcript_id": t.get("id"),              # ENST stable ID
            "name":          t.get("display_name"),    # e.g. "TP53-201"
            "biotype":       t.get("biotype"),          # e.g. "protein_coding"
            # length = transcript span on the genome (end - start + 1), not mRNA length
            "length":        (t.get("end", 0) - t.get("start", 0) + 1),
            "exon_count":    len(exons),
            # is_canonical: Ensembl marks one transcript per gene as canonical
            "is_canonical":  bool(t.get("is_canonical", 0)),
        })

    return pl.DataFrame(rows).with_columns([
        pl.col("length").cast(pl.Int64),
        pl.col("exon_count").cast(pl.Int32),
    ]).sort("length", descending=True)   # longest transcript first


# Extract the TP53 record from the cached list
tp53_record = next(g for g in cancer_genes_raw if g.get("display_name") == "TP53")

tp53_transcripts_df = parse_transcripts(tp53_record)

print(f"Shape : {tp53_transcripts_df.shape}")
print(f"\ndtypes:")
print(tp53_transcripts_df.dtypes)
print()
tp53_transcripts_df.head(10)